In [1]:
import openai
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

In [2]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

### Download all data from Qdrant

In [ ]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [ ]:
all_points

In [ ]:
all_context = [
    {"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]
]

In [ ]:
all_context

#### Render a prompt to generate synthetic Eval reference dataset

In [ ]:
output_schema = {
    "type": "array",
    "items":{
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question"
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "Suggested answer grounded in the content."
                }
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks."
            }
        }
    }
}


SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come and ...


<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""


USER_PROMPT = f"""
    Here is the list of chucks, each list element is a dictionary with id and text:
    {all_context}
"""



In [ ]:
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content":USER_PROMPT}
    ],
    reasoning_effort="minimal"
)


print(response.choices[0].message.content)

In [ ]:
import json


json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [ ]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="BOBYYLJRHT")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [ ]:
print(points[0].payload)

In [ ]:
def get_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-00",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]
    


In [ ]:
get_description("BOJDJDS0")

### Create Eval dataset in Langsmith

In [ ]:
client = Client(api_key=os.environ["LANSMITH_API_KEY"])

In [ ]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline"
)

In [ ]:
for item in json_output:
    print(item["chunck_ids"])
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_description": [get_description(id) for id in item["chunk_ids"]]
        }
    )